In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!rm -rf /content/xai_subset
!unzip -oq subset_3000_for_colab.zip -d /content/xai_subset

In [ ]:
import os
import pandas as pd
from pathlib import Path

metadata_path = "/content/xai_subset/outputs_3000/selected_metadata_3000_colab.csv"
image_dir = "/content/xai_subset/subset_3000_images"

df = pd.read_csv(metadata_path)
df["colab_image_path"] = df["local_image_path"].apply(lambda x: os.path.join(image_dir, x))

print("Metadata shape:", df.shape)
print("Existing files:", df["colab_image_path"].apply(lambda p: Path(p).exists()).mean())
print(df["view"].value_counts())
display(df.head())

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q open_clip_torch

In [ ]:
import open_clip
from PIL import Image
from tqdm import tqdm
import numpy as np

model, _, preprocess = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)

model = model.to(device)
model.eval()

In [ ]:
batch_size = 32
all_embeddings = []

image_paths = df["colab_image_path"].tolist()

for start in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[start:start + batch_size]

    images = [
        preprocess(Image.open(p).convert("RGB"))
        for p in batch_paths
    ]

    batch = torch.stack(images).to(device)

    with torch.no_grad():
        feats = model.encode_image(batch)
        feats = feats / feats.norm(dim=-1, keepdim=True)

    all_embeddings.append(feats.cpu().numpy())

image_embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

print("Embeddings shape:", image_embeddings.shape)

In [ ]:
import os

output_dir = "/content/xai_subset/outputs_3000_biomedclip"
os.makedirs(output_dir, exist_ok=True)

np.save(
    os.path.join(output_dir, "image_embeddings_3000_biomedclip.npy"),
    image_embeddings
)

df[["image_id", "local_image_path", "colab_image_path", "view", "subject_id"]].to_csv(
    os.path.join(output_dir, "metadata_with_embeddings_3000.csv"),
    index=False
)

print(os.listdir(output_dir))

In [ ]:
!cd /content/xai_subset && zip -qr /content/outputs_with_embeddings_3000_biomedclip.zip outputs_3000_biomedclip

In [ ]:
from google.colab import files
files.download("/content/outputs_with_embeddings_3000_biomedclip.zip")

In [ ]:
output_dir = "/content/xai_subset/outputs"

np.save(os.path.join(output_dir, "image_embeddings_3000_biomedclip.npy"), image_embeddings)

df[["image_id", "local_image_path", "colab_image_path", "view", "subject_id"]].to_csv(
    os.path.join(output_dir, "metadata_with_embeddings_3000.csv"),
    index=False
)

print(os.listdir(output_dir))